In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'
# define schema for the BRAND data file 
brand_schema = StructType([
  StructField('brand_code', StringType(), False),
  StructField('brand_name', StringType(), True),
  StructField('category_code', StringType(), True),
])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/brands/*.csv"
#create a varirable, IN that whatever CSV I have I'll use it for bronze layer(may be)
df = spark.read.option("header", True).option("delimeter", ",").schema(brand_schema).csv(raw_data_path)

df = df.withColumn("source_file", F.col("_metadata.file_path")) \
      .withColumn("ingested_at", F.current_timestamp())
display(df.limit(5))

brand_code,brand_name,category_code,source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-06-04T07:37:57.139Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-06-04T07:37:57.139Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-06-04T07:37:57.139Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-06-04T07:37:57.139Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-06-04T07:37:57.139Z


In [0]:
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.bronze.brz_brands")

In [0]:
category_schema = StructType([
  StructField('category_code', StringType(), False),
  StructField('category_name', StringType(), True),
])

raw_data_path = "/Volumes/ecommerce/source_data/raw/category/*.csv"

df = spark.read.option("header", True).option("delimeter", ",").schema(category_schema).csv(raw_data_path)

df = df.withColumn("source_file", F.col("_metadata.file_path")) \
      .withColumn("ingested_at", F.current_timestamp())
display(df.limit(5))

category_code,category_name,source_file,ingested_at
ce,Electronics,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-04T07:38:12.104Z
app,Apparel,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-04T07:38:12.104Z
hnk,Home & Kitchen,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-04T07:38:12.104Z
bpc,Beauty & Personal Care,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-04T07:38:12.104Z
bks,Books,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-04T07:38:12.104Z


In [0]:
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.bronze.brz_category")

In [0]:
product_schema = StructType([
  StructField('product_id', StringType(), False),
  StructField('sku', StringType(), True),
  StructField('category_code', StringType(), True),
  StructField('brand_code', StringType(), True),
  StructField('color', StringType(), True),
  StructField('size', StringType(), True),
  StructField('material', StringType(), True),
  StructField('weight_grams', StringType(), True),  #Datatype str coz incoming data contain anomalities
  StructField('length_cm', StringType(), False),  #Datatype str coz incoming data contain anomalities
  StructField('width_cm', FloatType(), True),
  StructField('height_cm', FloatType(), True),
  StructField('rating_count', StringType(), True),
  StructField('File_name', StringType(), False),
  StructField('ingest_timestamp', TimestampType(), False),
 ])

raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.option("header", True).option("delimeter", ",").schema(product_schema).csv(raw_data_path)

df = df.withColumn("source_file", F.col("_metadata.file_path")) \
      .withColumn("ingested_at", F.current_timestamp())
display(df.limit(5))

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,File_name,ingest_timestamp,source_file,ingested_at
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-06-04T07:38:16.529Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-06-04T07:38:16.529Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-06-04T07:38:16.529Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-06-04T07:38:16.529Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-06-04T07:38:16.529Z


In [0]:
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.bronze.brz_products")

In [0]:
date_schema = StructType([
  StructField('date', StringType(), True), # raw date should be in str format
  StructField('year', IntegerType(), True), 
  StructField('day_name', StringType(), True), 
  StructField('quarter', IntegerType(), True),
  StructField('week_of_year', IntegerType(), True),
 ])

raw_data_path = "/Volumes/ecommerce/source_data/raw/date/*.csv"

df = spark.read.option("header", True).option("delimeter", ",").schema(date_schema).csv(raw_data_path)

df = df.withColumn("source_file", F.col("_metadata.file_path")) \
      .withColumn("ingested_at", F.current_timestamp())
display(df.limit(5))

df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.bronze.brz_date")

date,year,day_name,quarter,week_of_year,source_file,ingested_at
01-08-2025,2025,friday,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-06-04T07:38:21.132Z
02-08-2025,2025,SATURDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-06-04T07:38:21.132Z
03-08-2025,2025,SUNDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-06-04T07:38:21.132Z
04-08-2025,2025,MONDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-06-04T07:38:21.132Z
05-08-2025,2025,TUESDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-06-04T07:38:21.132Z


In [0]:

# Define ONLY the correct customer schema
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

raw_data_path = "/Volumes/ecommerce/source_data/raw/customers/*.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("delimiter", ",") \
    .schema(customer_schema) \
    .csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Overwrite — note: NO mergeSchema this time
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.bronze.brz_customer")